In [ ]:
# ==========================================================================
# 홈쇼핑 물류·배송 문의 유형 분류 웹 화면 (Gradio) — 셀 하나로 실행
# --------------------------------------------------------------------------
# [전체 흐름]
#   0. 도구 설치          : kiwipiepy, koreanize-matplotlib, gradio
#   1. 전처리 함수        : 대화 파일 → [S번호] 단위로 대화 나누기 → 중복 제거
#                           → 인사·대답을 건너뛴 '고객 질문 2마디' 뽑기 → Kiwi로 단어 나누기
#   2. 학습 함수 (train)  : 정답 파일과 합치기 → 학습 80% / 평가 20% 나누기
#                           → TF-IDF → 로지스틱 회귀(균형) 학습 → 성적표·혼동 행렬
#   3. 응답 함수 (ask)    : 새 문의 → 같은 방법으로 단어 나누기 → 유형별 확률(응답)
#                           → 과거 문의 중 가장 비슷한 5개 찾기(검색 결과)
#   4. 웹 화면 실행       : ① 파일 선택 → ② 학습 결과 → ③ 문의 분류·검색 순서로 배치
#
# [사용 순서]
#   ① 이 셀 실행 → 셀 아래(또는 https://….gradio.live 주소)에 웹 화면이 나타남
#   ② 상담 대화 .txt 파일과 정답 .csv 파일(열: ID, 유형)을 고르고 → [학습하기] (30초 정도)
#   ③ 고객 문의 문장을 입력하거나 예시를 누르고 → [분류하기] (Enter 가능)
#   끝낼 때: 셀 왼쪽의 ■(중지) 버튼
#
# [주의] 공개 주소(….gradio.live)는 주소를 아는 누구나 접속할 수 있음
# ==========================================================================


# ----- 0. 도구 설치 -----
# PIL(Pillow): Colab에 이미 설치된 이미지 도구, 지금 버전 번호를 알아내려고 불러옴
import PIL

# pip 설치 (-q: 메시지 줄이기)
# pillow=={PIL.__version__}: 설치 중에 Pillow 버전이 바뀌어 ImportError가 나는 것을 막으려고 지금 버전으로 고정
!pip install -q kiwipiepy koreanize-matplotlib gradio pillow=={PIL.__version__}


# ----- 도구 불러오기 -----
import re                                         # 정규식(글자 패턴 찾기·바꾸기)
import pandas as pd                               # 표(DataFrame) 다루기
import matplotlib.pyplot as plt                   # 그래프 그리기
import koreanize_matplotlib                       # 불러오기만 하면 그래프 한글 글꼴(나눔고딕)이 설정됨
from kiwipiepy import Kiwi                        # 한국어 형태소 분석기
from sklearn.model_selection import train_test_split               # 학습용/평가용 나누기
from sklearn.feature_extraction.text import TfidfVectorizer        # 단어 → TF-IDF 숫자
from sklearn.linear_model import LogisticRegression                # 분류 모델
from sklearn.metrics import accuracy_score, f1_score, classification_report, ConfusionMatrixDisplay
from sklearn.metrics.pairwise import cosine_similarity             # 문장끼리 비슷한 정도(0~1)
import gradio as gr                               # 웹 화면 만들기

# 한국어 형태소 분석기 준비 (처음 한 번 몇 초 걸림)
kiwi = Kiwi()

# 유형 순서 (표와 그래프에서 늘어놓을 순서)
ORDER = ["상품 주문", "배송 조회·지연", "주소·주문 변경", "취소·반품·교환", "결제·기타"]


# ----- 1. 전처리 함수 -----
# ✏️ 인사·맞장구·대답 표현 (발화에서 이 표현들을 모두 지웠을 때 남는 글자가 없으면 '의미 없는 발화')
#    | 는 '또는', [\s.?!,~] 는 공백이나 . ? ! , ~ 중 한 글자
#    긴 표현을 앞에 둬야 '맞습니다'가 '맞'+'습니다'로 쪼개지지 않고 통째로 지워짐
ETC = (r"안녕하세요|안녕하십니까|여보세요|수고하세요|수고하십니다|수고하십시오|수고 많으십니다|반갑습니다"
       r"|알겠습니다|알겠어요|알았어요|알았습니다|맞습니다|맞아요|맞네요|그래요|그렇죠|그렇군요|그렇습니다"
       r"|감사합니다|감사해요|고맙습니다|고마워요|괜찮아요|괜찮습니다|없습니다|없어요|아니요|아니오|아니에요"
       r"|잠시만요|잠깐만요|입니다|이구요|이고요|이에요|이예요|이요|이고|에요|예요|네|예|아|음|요|[\s.?!,~]")

# ✏️ 이어 붙일 고객 발화 수 (1로 바꾸면 '첫 질문 한 마디'만 사용)
N_UTTER = 2

# ✏️ 남길 품사: NNG 일반명사, NNP 고유명사, VV 동사, VA 형용사, XR 어근, MAG 부사, SL 외국어
KEEP = {"NNG", "NNP", "VV", "VA", "XR", "MAG", "SL"}


# 대화 하나 → 의미 있는 고객 발화 N_UTTER개를 이어 붙인 문장
def customer_question(dialog):
    found = []                                         # 찾은 발화를 담을 빈 목록
    for line in dialog.splitlines():                   # 대화를 한 줄씩 꺼내기
        if not line.startswith("고객:"):
            continue                                   # 상담사 발화는 건너뛰기
        q = line[3:]                                   # 앞의 '고객:' 세 글자 떼기
        q = re.sub(r"ㅇ+|O+", "", q)                   # 가림 표시(ㅇㅇㅇ, OOO) 지우기
        q = re.sub(r"\s+", " ", q).strip()             # 여러 칸 공백을 한 칸으로, 앞뒤 공백 지우기
        if len(q) >= 5 and re.sub(ETC, "", q) != "":   # 5글자 이상이고, 인사·대답을 지워도 글자가 남으면
            found.append(q)                            # 의미 있는 발화로 보고 목록에 추가
        if len(found) == N_UTTER:                      # 필요한 개수를 다 찾으면
            break                                      # 반복 멈추기
    return " ".join(found)                             # 찾은 발화를 한 칸 띄워서 이어 붙이기


# 문장 하나 → 남길 품사의 단어만 골라 띄어쓰기로 이은 글자
#   t.form: 형태소 글자, t.tag.split("-")[0]: 'VA-I' → 'VA' 처럼 '-' 앞부분만 사용
def to_words(sentence):
    return " ".join(t.form for t in kiwi.tokenize(sentence) if t.tag.split("-")[0] in KEEP)


# 대화 파일 글자 전체 → [ID, 원문, 질문, 단어] 표
def make_table(text):
    # '[S숫자]' 줄을 기준으로 자르기 → ['', 'S20007', '대화 내용', 'S19995', '대화 내용', ...]
    #   (S\d+): 번호도 결과에 남김, flags=re.M: ^와 $를 줄마다 적용
    parts = re.split(r"^\[(S\d+)\]\s*$", text, flags=re.M)
    # parts[1::2] = 번호들, parts[2::2] = 대화 내용들 → 두 열짜리 표
    df = pd.DataFrame({"ID": parts[1::2], "원문": [p.strip() for p in parts[2::2]]})
    # 완전히 같은 대화 지우기 (reset_index: 줄 번호를 0부터 다시 매김)
    df = df.drop_duplicates("원문").reset_index(drop=True)
    # 대화마다 고객 질문 2마디, 그 질문의 단어 목록 (단어 나누기에 20초 정도 걸림)
    df["질문"] = df["원문"].apply(customer_question)
    df["단어"] = df["질문"].apply(to_words)
    return df


# ----- 2. 학습 함수: [학습하기] 버튼 -----
def train(txt_file, label_file):
    # 파일 두 개를 다 고르지 않았으면 안내 메시지만 돌려줌 (나머지 출력은 비움)
    if txt_file is None or label_file is None:
        return "상담 대화 .txt 파일과 정답 .csv 파일을 모두 선택하세요.", None, None, None

    # 대화 파일 읽기 → 전처리 표
    df = make_table(open(txt_file, encoding="utf-8").read())

    # 정답 파일(열: ID, 유형)과 합치기: 두 표에 모두 있는 ID만 남김 (how="inner")
    # 질문 글자가 똑같은 문의는 하나만 남김 (같은 문장이 학습·평가에 동시에 들어가 점수가 부풀려지는 것을 막음)
    data = df.merge(pd.read_csv(label_file), on="ID", how="inner")
    data = data.drop_duplicates("질문").reset_index(drop=True)
    if len(data) == 0:
        return "대화 번호(ID)와 정답 파일이 맞지 않습니다. 파일을 확인하세요.", None, None, None

    # 80%는 학습용, 20%는 평가용 (stratify: 유형 비율을 똑같이, random_state: 매번 같게)
    X_train, X_test, y_train, y_test = train_test_split(
        data["단어"], data["유형"], test_size=0.2, random_state=42, stratify=data["유형"])

    # TF-IDF: 학습용으로 단어 사전을 만들고(fit) 숫자로 바꾸기, 평가용은 바꾸기만
    # (analyzer=str.split: 이미 띄어쓰기로 나눠 둔 단어를 그대로 사용)
    tfidf = TfidfVectorizer(analyzer=str.split)
    A_train = tfidf.fit_transform(X_train)
    A_test = tfidf.transform(X_test)

    # 로지스틱 회귀 학습 (class_weight="balanced": 개수가 적은 유형을 틀리면 더 크게 벌점)
    model = LogisticRegression(max_iter=1000, class_weight="balanced").fit(A_train, y_train)

    # 평가용으로 예측하고 정확도·macro-F1 계산 (macro-F1: 유형마다 F1을 구해 똑같은 비중으로 평균)
    pred = model.predict(A_test)
    acc = accuracy_score(y_test, pred)
    f1 = f1_score(y_test, pred, average="macro")

    # 유형별 성적표 (output_dict=True: 사전으로 받아 표로 바꿈, .T: 행과 열 바꾸기)
    # → ORDER 유형만 남기고 소수 셋째 자리로 반올림, 열 이름을 한글로
    report = pd.DataFrame(classification_report(y_test, pred, labels=ORDER, output_dict=True)).T
    report = report.loc[ORDER].round(3).reset_index().rename(columns={"index": "유형", "support": "평가 문의 수"})
    report["평가 문의 수"] = report["평가 문의 수"].astype(int)            # 332.0 → 332 (소수점 없애기)

    # 혼동 행렬 그림 (가로 = 예측, 세로 = 정답, 대각선 = 맞힌 문의 수)
    fig, ax = plt.subplots(figsize=(7, 6))
    ConfusionMatrixDisplay.from_predictions(y_test, pred, labels=ORDER, cmap="Blues",
                                            colorbar=False, ax=ax, xticks_rotation=30)
    ax.set_xlabel("예측")
    ax.set_ylabel("정답")
    ax.set_title("혼동 행렬 (평가용)")
    plt.tight_layout()                                     # 글자가 잘리지 않게 여백 자동 조정
    plt.close(fig)                                         # 셀 아래에 따로 그려지지 않게 닫기 (웹 화면에만 표시)

    # 처리 결과 메시지 ({:,}: 천 단위 쉼표, {:.1%}: 백분율 소수 첫째 자리)
    message = (f"문의 {len(data):,}건으로 학습 완료 (학습 {len(X_train):,} / 평가 {len(X_test):,}). "
               f"평가용 정확도 {acc:.1%}, macro-F1 {f1:.3f}")

    # 검색용: 전체 문의를 TF-IDF 숫자로 바꿔 둠
    A_all = tfidf.transform(data["단어"])

    # 메시지, 성적표, 혼동 행렬, 보관함(응답 함수가 쓸 도구와 데이터 묶음)을 돌려줌
    return message, report, fig, (tfidf, model, data, A_all)


# ----- 3. 응답 함수: [분류하기] 버튼 -----
def ask(question, store):
    # 학습 전이거나 문의가 비어 있으면 안내 메시지
    if store is None:
        return "먼저 파일 두 개를 고르고 [학습하기]를 누르세요.", None, None
    if not question.strip():
        return "고객 문의 문장을 입력하세요.", None, None

    # 보관함에서 TF-IDF 도구, 모델, 데이터, 전체 문의 숫자 행렬 꺼내기
    tfidf, model, data, A_all = store

    # 새 문의도 학습 때와 같은 방법으로 단어 나누기 → TF-IDF 숫자로 바꾸기 (fit 없이 transform만)
    words = to_words(question)
    A_new = tfidf.transform([words])

    # 유형별 확률 (predict_proba: 0~1 → ×100 해서 %), 모델의 유형 순서를 ORDER 순서로 다시 정렬
    proba = pd.Series(model.predict_proba(A_new)[0] * 100, index=model.classes_)[ORDER]
    best = proba.idxmax()                                  # 확률이 가장 큰 유형

    # 응답 글
    answer = (f"예측 유형: {best} ({proba[best]:.1f}%)\n"
              f"분석에 쓴 단어: {words if words else '(없음 — 분석할 단어가 없어 예측이 불확실함)'}")

    # 유형별 확률 표
    proba_table = proba.round(1).reset_index()
    proba_table.columns = ["유형", "확률(%)"]

    # 검색 결과: 새 문의와 TF-IDF 코사인 유사도가 가장 높은 과거 문의 5개
    scores = cosine_similarity(A_new, A_all)[0]            # 과거 문의마다 비슷한 정도(0~1)
    top = scores.argsort()[::-1][:5]                       # 유사도 큰 순서로 5개 위치
    top = top[scores[top] > 0]                             # 겹치는 단어가 하나도 없는(유사도 0) 문의는 빼기
    similar = data.loc[top, ["ID", "질문", "유형"]].assign(유사도=scores[top].round(3))

    return answer, proba_table, similar


# ----- 4. 웹 화면 -----
# gr.Blocks(): 웹 화면 전체 틀, with 블록 안의 부품이 위에서부터 차례로 배치됨
with gr.Blocks() as demo:
    gr.Markdown("# 📦 홈쇼핑 물류·배송 문의 유형 분류")
    gr.Markdown("고객이 처음 꺼낸 **말 두 마디**로 문의 유형(상품 주문 / 배송 조회·지연 / 주소·주문 변경 / "
                "취소·반품·교환 / 결제·기타)을 자동으로 분류하고, 비슷한 과거 문의를 찾아 줍니다. "
                "(Kiwi + TF-IDF + 로지스틱 회귀)")

    # 보이지 않는 보관함: train이 만든 (TF-IDF, 모델, 데이터, 숫자 행렬)을 담아 ask에 넘겨줌
    store = gr.State()

    # ===== ① 파일 선택 =====
    gr.Markdown("## ① 파일 선택")
    with gr.Row():                                         # gr.Row(): 안의 부품을 가로로 나란히
        txt_file = gr.File(label="상담 대화 파일 (.txt)", file_types=[".txt"])
        label_file = gr.File(label="정답 파일 (.csv, 열: ID, 유형)", file_types=[".csv"])
    train_btn = gr.Button("학습하기", variant="primary")
    status = gr.Textbox(label="처리 결과")

    # ===== ② 학습 결과 =====
    gr.Markdown("## ② 학습 결과 (평가용 20%)")
    report = gr.Dataframe(label="유형별 성적표 (precision 정밀도 · recall 재현율 · f1-score)")
    cm_plot = gr.Plot(label="혼동 행렬")

    # ===== ③ 문의 분류와 비슷한 문의 검색 =====
    gr.Markdown("## ③ 문의 분류하기")
    question = gr.Textbox(label="고객 문의", placeholder="예) 지난주에 주문했는데 아직도 안 왔어요")
    # ✏️ 예시 문의: 누르면 입력 칸이 채워짐
    gr.Examples(examples=["지금 방송하는 갈비 주문하려고요", "지난주에 주문했는데 아직도 안 왔어요",
                          "어제 주문한 거 배송지를 바꾸고 싶어요", "신발이 작아서 한 사이즈 큰 걸로 교환하고 싶어요",
                          "무통장 입금했는데 확인이 안 된다고 나와요"], inputs=question)
    ask_btn = gr.Button("분류하기", variant="primary")
    answer = gr.Textbox(label="응답", lines=2)
    with gr.Row():
        proba_table = gr.Dataframe(label="유형별 확률", scale=1)
        similar = gr.Dataframe(label="검색 결과: 비슷한 과거 문의 5개", wrap=True, scale=3,
                               column_widths=["12%", "56%", "20%", "12%"])   # 열 너비 (ID, 질문, 유형, 유사도)

    # 버튼과 함수 연결: 함수가 돌려준 값이 outputs 부품에 순서대로 표시됨
    train_btn.click(train, inputs=[txt_file, label_file], outputs=[status, report, cm_plot, store])
    ask_btn.click(ask, inputs=[question, store], outputs=[answer, proba_table, similar])
    question.submit(ask, inputs=[question, store], outputs=[answer, proba_table, similar])   # Enter로도 실행

# 웹 화면 실행 (debug=True: 셀이 계속 실행되며 오류를 셀 아래에 보여 줌)
demo.launch(debug=True)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 MB 4.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 28.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.9/7.9 MB 42.0 MB/s eta 0:00:00
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://f6487a8fa60715bb1e.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (http